<a href="https://colab.research.google.com/github/barrevivo299-design/SMS-Spam-Classification-NaiveBayes/blob/main/SMS_Spam_Classification_NaiveBayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Assignment – Text Classification (NLP)
## Multinomial Naive Bayes – Implementation from Scratch

| Field | Details |
| :--- | :--- |
| **Student** | Bar Revivo \| Last 4 digits of ID: 0076 |
| **Assignment Type** | Text Analysis (NLP) |
| **Learning Type** | Binary Classification (`spam` vs. `ham`) |
| **Algorithm Implemented** | Multinomial Naive Bayes (Implemented from Scratch) |
| **Evaluation Metric** | F1-Score on the `spam` class |
| **Dataset** | [SMS Spam Collection Dataset on Kaggle](https://www.kaggle.com/datasets/hamnawaseem112222222/sms-spam-collection-5572-labeled-sms-messages) |

---

## 1. Prompts, Tools & Additional Sources

Below are the key prompts and resources used throughout the assignment.

### Prompts Used
1. **Prompt:** "Explain how to structure a machine learning pipeline for text classification using Naive Bayes from scratch."
   - **Purpose:** Understanding the workflow and steps required for NLP data preparation.
2. **Prompt:** "How to convert raw text messages into a word frequency matrix without data leakage?"
   - **Purpose:** Implementing Feature Engineering correctly on Train and Test sets.

### Additional Sources
* [scikit-learn Naive Bayes Guide](https://scikit-learn.org/stable/modules/naive_bayes.html)
* [scikit-learn CountVectorizer Reference](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
* [Kaggle SMS Dataset](https://www.kaggle.com/datasets/hamnawaseem112222222/sms-spam-collection-5572-labeled-sms-messages)

---

## 2. Problem Overview & Dataset Description

### Problem Statement
The objective is to build a binary classifier to predict whether an incoming text message is **Spam** (1) or **Ham** (0).

### Dataset Details
* **Source:** SMS Spam Collection Dataset containing 5,572 labeled messages.
* **Class Imbalance:** Mostly legitimate messages (`ham` about 86.6%) with a minority of `spam` (about 13.4%).
* **Evaluation:** Due to imbalance, the model is evaluated using the **F1-Score on the `spam` class**.

In [24]:
import pandas as pd
import numpy as np

# 1. Load Dataset directly from source
url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['Category', 'Message'])

# Display Dataset shape & Class distribution
print(f"גודל ה-dataset: {df.shape}\n")
print("התפלגות המחלקות:")
print(df['Category'].value_counts())
print()

# Display the first 5 rows as a rich Pandas Table
df.head()

גודל ה-dataset: (5572, 2)

התפלגות המחלקות:
Category
ham     4825
spam     747
Name: count, dtype: int64



,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## Data Preparation: Deduplication and Single Split

The dataset is provided as a single file and is not pre-split into training and test sets. Therefore, we perform a **single and fixed split** so that all generated rows apply consistently to both subsets throughout the project. No further splits will be performed.

### Key Considerations
* **Data Leakage Prevention:** Duplicate rows are removed prior to splitting. This order is critical: splitting before deduplication could allow identical messages to appear in both the training and testing sets, causing data leakage that artificially inflates performance metrics.
* **Stratified Split:** We use a Stratified split to maintain the class distribution balance across both subsets, along with a fixed `random_state` for full reproducibility.

In [25]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Removed {before - len(df)} duplicate rows. {len(df)} rows remain.')

df['label'] = (df['Category'] == 'spam').astype(int)

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=RANDOM_STATE
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'\ntrain: {train_df.shape}   test: {test_df.shape}')
print(f'spam ratio in train: {100*train_df.label.mean():.2f}%')
print(f'spam ratio in test : {100*test_df.label.mean():.2f}%')

print('\nFirst 5 rows of the trainset:')
display(train_df[['Category', 'Message', 'label']].head())

print('First 5 rows of the test-set:')
display(test_df[['Category', 'Message', 'label']].head())

Removed 403 duplicate rows. 5169 rows remain.

train: (4135, 3)   test: (1034, 3)
spam ratio in train: 12.62%
spam ratio in test : 12.67%

First 5 rows of the trainset:


,Category,Message,label
0,ham,"Ta-Daaaaa! I am home babe, are you still up ?",0
1,ham,Yup having my lunch buffet now.. U eat already?,0
2,ham,Ok anyway no need to change with what you said,0
3,ham,All e best 4 ur exam later.,0
4,ham,Now only i reached home. . . I am very tired n...,0


First 5 rows of the test-set:


,Category,Message,label
0,ham,"Good morning, my Love ... I go to sleep now an...",0
1,ham,"And how you will do that, princess? :)",0
2,ham,"Cool, I'll text you when I'm on the way",0
3,ham,Your right! I'll make the appointment right now.,0
4,ham,Aiya we discuss later lar... Pick ü up at 4 is...,0


## Evaluation Metric

For binary classification problems with a single focus class—such as identifying spam messages (`spam` / `label=1`)—we evaluate model performance using the **F1-Score calculated strictly for the primary class (`spam`)**.

### Metric Justification
* **Class Imbalance:** The dataset is imbalanced (~86% ham vs. ~14% spam). A naive baseline classifier predicting all messages as `ham` would achieve ~86% Accuracy while failing to catch a single spam message. Accuracy is therefore misleading.
* **Balanced Trade-off:** F1-Score represents the harmonic mean of Precision and Recall. It ensures that our model minimizes both false positives (legitimate messages flagged as spam) and false negatives (spam messages missed by the filter).

In [26]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, f1_score
import numpy as np

vectorizer = CountVectorizer(stop_words='english', lowercase=True)
X_train = vectorizer.fit_transform(train_df['Message']).toarray()
X_test = vectorizer.transform(test_df['Message']).toarray()

y_train = train_df['label'].values
y_test = test_df['label'].values

class MultinomialNaiveBayesFromScratch:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)

        self.priors = np.bincount(y) / float(n_samples)
        self.feature_counts = np.zeros((n_classes, n_features))
        for c in self.classes:
            self.feature_counts[c] = X[y == c].sum(axis=0)

        self.class_word_counts = self.feature_counts.sum(axis=1)

    def predict(self, X):
        log_probs = []
        for c in self.classes:
            smoothed_word_probs = (self.feature_counts[c] + self.alpha) / (self.class_word_counts[c] + self.alpha * X.shape[1])
            log_likelihood = X @ np.log(smoothed_word_probs)
            log_prior = np.log(self.priors[c])
            log_probs.append(log_prior + log_likelihood)

        log_probs = np.array(log_probs).T
        return np.argmax(log_probs, axis=1)

model = MultinomialNaiveBayesFromScratch(alpha=1.0)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
spam_f1 = f1_score(y_test, y_pred, pos_label=1)

print(f"F1-Score on Spam Class (pos_label=1): {spam_f1:.4f}\n")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

F1-Score on Spam Class (pos_label=1): 0.9333

              precision    recall  f1-score   support

         Ham       0.99      0.99      0.99       903
        Spam       0.96      0.91      0.93       131

    accuracy                           0.98      1034
   macro avg       0.97      0.95      0.96      1034
weighted avg       0.98      0.98      0.98      1034



## Feature Engineering

The raw text is transformed into a numerical representation using a five-step processing pipeline:

1. **Text Cleaning & Normalization:** Converting text to lowercase and replacing specific entities with generic tokens (URLs, phone numbers, currency symbols).  
   > **Rationale:** A specific phone number appears only once in the corpus and holds no predictive value, but the presence of *any* phone number is a strong indicator of spam.
2. **Tokenization:** Splitting the text into individual words/tokens.
3. **Stopwords Removal:** Eliminating common words like `at`, `is`, or `the` that appear with similar frequencies across both classes and carry no discriminative information.
4. **Stemming (Porter Stemmer):** Mapping word inflections to a shared root form (`win`, `winning`, `winner` $\rightarrow$ `win`). This reduces vocabulary size and reinforces word frequencies.
5. **Bag of Words:** Constructing a vector representation based on word occurrence counts.

### Data Leakage Prevention
The vocabulary is built using `fit_transform` exclusively on the training set, while only `transform` is applied to the test set. Any word appearing solely in the test set is excluded from the vocabulary—simulating a real-world scenario where the deployed model encounters unseen messages after training.

In [27]:
import re
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer

STOPWORDS = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(t):
    t = t.lower()
    t = re.sub(r'http\S+|www\.\S+', ' urltoken ', t)
    t = re.sub(r'[\w\.-]+@[\w\.-]+\b', ' emailtoken ', t)
    t = re.sub(r'[£$€]', ' currencytoken ', t)
    t = re.sub(r'\b\d{5,}\b', ' phonetoken ', t)
    t = re.sub(r'\d+', ' numtoken ', t)
    t = re.sub(r'[^a-z\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def tokenize(t):
    return t.split()

def remove_stopwords(toks):
    return [w for w in toks if w not in STOPWORDS and len(w) > 1]

def stem(toks):
    return [stemmer.stem(w) for w in toks]

def preprocess(t):
    return ' '.join(stem(remove_stopwords(tokenize(clean_text(t)))))

train_df['clean'] = train_df['Message'].apply(preprocess)
test_df['clean'] = test_df['Message'].apply(preprocess)

In [28]:
def show_pipeline(d, name, n=3):
    print('=' * 60)
    print(f'PIPELINE DEMO - {name}')
    print('=' * 60)
    idx = d.index[d['label'] == 1][:1].tolist() + d.index[d['label'] == 0][:2].tolist()
    for i in idx[:n]:
        raw = d.loc[i, 'Message']
        print(f'\n[label = {"spam" if d.loc[i, "label"]==1 else "ham"}]')
        print('1. raw          :', raw[:100])
        print('2. cleaned      :', clean_text(raw)[:100])
        print('3. tokens       :', tokenize(clean_text(raw))[:10])
        print('4. no stopwords :', remove_stopwords(tokenize(clean_text(raw)))[:10])
        print('5. stemmed      :', stem(remove_stopwords(tokenize(clean_text(raw))))[:10])

show_pipeline(train_df, 'TRAIN')
show_pipeline(test_df, 'TEST')

PIPELINE DEMO - TRAIN

[label = spam]
1. raw          : 18 days to Euro2004 kickoff! U will be kept informed of all the latest news and results daily. Unsub
2. cleaned      : numtoken days to euro numtoken kickoff u will be kept informed of all the latest news and results da
3. tokens       : ['numtoken', 'days', 'to', 'euro', 'numtoken', 'kickoff', 'u', 'will', 'be', 'kept']
4. no stopwords : ['numtoken', 'days', 'euro', 'numtoken', 'kickoff', 'kept', 'informed', 'latest', 'news', 'results']
5. stemmed      : ['numtoken', 'day', 'euro', 'numtoken', 'kickoff', 'kept', 'inform', 'latest', 'news', 'result']

[label = ham]
1. raw          : Ta-Daaaaa! I am home babe, are you still up ?
2. cleaned      : ta daaaaa i am home babe are you still up
3. tokens       : ['ta', 'daaaaa', 'i', 'am', 'home', 'babe', 'are', 'you', 'still', 'up']
4. no stopwords : ['ta', 'daaaaa', 'home', 'babe', 'still']
5. stemmed      : ['ta', 'daaaaa', 'home', 'babe', 'still']

[label = ham]
1. raw          : Yup 

In [29]:
vectorizer = CountVectorizer(min_df=2, ngram_range=(1, 1))

X_train = vectorizer.fit_transform(train_df['clean'])
X_test = vectorizer.transform(test_df['clean'])

y_train = train_df['label'].values
y_test = test_df['label'].values

print(f'Vocabulary size: {len(vectorizer.vocabulary_)}')
print(f'X_train shape: {X_train.shape} | X_test shape: {X_test.shape}')
print(f'Sparsity Density: {100 * X_train.nnz / (X_train.shape[0] * X_train.shape[1]):.4f}%')

Vocabulary size: 2573
X_train shape: (4135, 2573) | X_test shape: (1034, 2573)
Sparsity Density: 0.2955%


## Multinomial & Bernoulli Naive Bayes Implementation

### Bayes' Theorem
For a message $d$ and class $c \in \{\text{ham}, \text{spam}\}$:

$$P(c \mid d) \propto P(c) \cdot P(d \mid c)$$

To classify a message, it is sufficient to compare the products of the class prior probability and the likelihood of observing the message given that class. Since the marginal probability denominator $P(d)$ is identical across all classes, it can be safely omitted.

---

### Naive Assumption
Assuming words are conditionally independent given the class:

$$P(d \mid c) = \prod_{i=1}^{n} P(w_i \mid c)^{\text{count}(w_i, d)}$$

While assuming word independence is technically incorrect for natural language, it simplifies a joint distribution over thousands of dimensions into single-dimensional conditional probabilities, working remarkably well in practice.

---

### Laplace Smoothing
Estimating word likelihoods with additive smoothing:

$$P(w \mid c) = \frac{\text{count}(w, c) + \alpha}{\sum_{w'} \text{count}(w', c) + \alpha \cdot |V|}$$

The hyperparameter $\alpha$ prevents zero probabilities. Without smoothing ($\alpha = 0$), encountering an unseen word in a class sets its probability to zero, invalidating the entire product. Smoothing ensures every word maintains a non-zero probability baseline.

---

### Log-Space Transformation
Converting probability products into logarithmic sums:

$$\log P(c \mid d) \propto \log P(c) + \sum_{i=1}^{n} \text{count}(w_i, d) \cdot \log P(w_i \mid c)$$

Multiplying hundreds of small probabilities causes floating-point underflow (rounding down to zero). Since the logarithm is a monotonically increasing function, maximizing the log-probability yields the exact same class prediction while ensuring numerical stability.

---

### Model Variants

| Variant | Input Data Representation | Typical Use Case |
| :--- | :--- | :--- |
| **Multinomial** | Word frequencies/counts | Text classification where term frequency carries signal |
| **Bernoulli** | Binary word presence/absence (0 or 1) | Short texts, where presence of specific words matters |

In [30]:
import numpy as np
from scipy import sparse

class CustomNaiveBayes:
    """
    Custom implementation of Naive Bayes classifier from scratch.
    Supports both Multinomial and Bernoulli variants with Laplace smoothing.
    """
    def __init__(self, alpha=1.0, variant='multinomial', fit_prior=True):
        if variant not in ('multinomial', 'bernoulli'):
            raise ValueError("variant must be either 'multinomial' or 'bernoulli'")

        self.alpha = float(alpha)
        self.variant = variant
        self.fit_prior = fit_prior

    def fit(self, X, y):
        X = sparse.csr_matrix(X, dtype=np.float64)

        if self.variant == 'bernoulli':
            X = (X > 0).astype(np.float64)

        self.classes_ = np.unique(y)
        n_samples, n_features = X.shape
        n_classes = len(self.classes_)

        self.class_log_prior_ = np.zeros(n_classes)
        self.feature_log_prob_ = np.zeros((n_classes, n_features))

        if self.variant == 'bernoulli':
            self.feature_log_prob_neg_ = np.zeros((n_classes, n_features))

        for i, c in enumerate(self.classes_):
            X_c = X[y == c]
            n_c = X_c.shape[0]

            # Compute Prior Log Probabilities
            if self.fit_prior:
                self.class_log_prior_[i] = np.log(n_c / float(n_samples))
            else:
                self.class_log_prior_[i] = np.log(1.0 / n_classes)

            # Compute Feature Likelihoods with Laplace Smoothing
            feature_counts = np.asarray(X_c.sum(axis=0)).ravel()

            if self.variant == 'multinomial':
                smoothed_numerator = feature_counts + self.alpha
                smoothed_denominator = feature_counts.sum() + (self.alpha * n_features)
                self.feature_log_prob_[i] = np.log(smoothed_numerator / smoothed_denominator)
            else:  # Bernoulli
                prob = (feature_counts + self.alpha) / (n_c + 2.0 * self.alpha)
                self.feature_log_prob_[i] = np.log(prob)
                self.feature_log_prob_neg_[i] = np.log(1.0 - prob)

        return self

    def _joint_log_likelihood(self, X):
        X = sparse.csr_matrix(X, dtype=np.float64)

        if self.variant == 'bernoulli':
            X = (X > 0).astype(np.float64)
            # Log likelihood incorporating both present and absent features
            joint_ll = X @ (self.feature_log_prob_ - self.feature_log_prob_neg_).T
            joint_ll += self.feature_log_prob_neg_.sum(axis=1)
        else:
            joint_ll = X @ self.feature_log_prob_.T

        return joint_ll + self.class_log_prior_

    def predict(self, X):
        jll = self._joint_log_likelihood(X)
        return self.classes_[np.argmax(jll, axis=1)]

    def predict_proba(self, X):
        jll = self._joint_log_likelihood(X)
        # Numerical stability using Log-Sum-Exp trick
        max_log = np.max(jll, axis=1, keepdims=True)
        exp_jll = np.exp(jll - max_log)
        return exp_jll / exp_jll.sum(axis=1, keepdims=True)

## Implementation Correctness Verification

Before hyperparameter tuning, we verify that our custom Naive Bayes implementation behaves identically to `scikit-learn`'s standard implementation (`MultinomialNB` and `BernoulliNB`).

> **Sanity Check Notice:** This verification is performed exclusively on the **train set** to validate implementation correctness without touching the test set.

In [31]:
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.metrics import f1_score
import numpy as np

for variant, ref_model in [('multinomial', MultinomialNB), ('bernoulli', BernoulliNB)]:
    mine = CustomNaiveBayes(alpha=1.0, variant=variant).fit(X_train, y_train)
    ref = ref_model(alpha=1.0).fit(X_train, y_train)

    p_mine, p_ref = mine.predict(X_train), ref.predict(X_train)
    same_proba = np.allclose(mine.predict_proba(X_train), ref.predict_proba(X_train))

    agreement = 100 * np.mean(p_mine == p_ref)
    train_f1 = f1_score(y_train, p_mine, pos_label=1)

    print(f"{variant:12s} | agreement={agreement:.2f}% | identical predict_proba={same_proba} | train f1={train_f1:.4f}")

multinomial  | agreement=100.00% | identical predict_proba=True | train f1=0.9612
bernoulli    | agreement=100.00% | identical predict_proba=True | train f1=0.9625


## Model Training & Hyperparameter Tuning (Cross-Validation)

In this step, we select the optimal combination of text representation vectorizer parameters and Naive Bayes hyperparameters using **5-Fold Stratified Cross-Validation** evaluated exclusively on the **train set**.

> **Methodological Note:** To avoid **Data Leakage**, the vectorizer is re-fitted inside each cross-validation fold using only that fold's training split.

### Search Space Parameters

| Parameter | Values | Description |
| :--- | :--- | :--- |
| `vec_type` | `count`, `tfidf` | Token frequency count vs. Term Frequency-Inverse Document Frequency |
| `ngram_range` | `(1, 1)`, `(1, 2)` | Unigrams only vs. Unigrams + Bigrams |
| `min_df` | `1`, `2` | Minimum document frequency threshold |
| `alpha` | `0.05`, `0.1`, `0.5`, `1.0` | Laplace smoothing intensity |
| `variant` | `multinomial`, `bernoulli` | Model distribution variant |

**Total Configurations:** $2 \times 2 \times 2 \times 4 \times 2 = 64$ combinations across 5 folds.

In [32]:
import itertools
import time
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

param_grid = {
    'vec_type': ['count', 'tfidf'],
    'ngram_range': [(1, 1), (1, 2)],
    'min_df': [1, 2],
    'alpha': [0.05, 0.1, 0.5, 1.0],
    'variant': ['multinomial', 'bernoulli']
}

keys = list(param_grid)
combos = [dict(zip(keys, v)) for v in itertools.product(*param_grid.values())]
print(f'Total combinations to evaluate: {len(combos)}')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
texts = train_df['clean'].values

results = []
t0 = time.time()

for params in combos:
    fold_scores = []
    for tr_idx, va_idx in skf.split(texts, y_train):
        Vec = CountVectorizer if params['vec_type'] == 'count' else TfidfVectorizer
        vec = Vec(ngram_range=params['ngram_range'], min_df=params['min_df'])

        X_tr = vec.fit_transform(texts[tr_idx])
        X_va = vec.transform(texts[va_idx])

        model = CustomNaiveBayes(alpha=params['alpha'], variant=params['variant'])
        model.fit(X_tr, y_train[tr_idx])

        preds = model.predict(X_va)
        fold_scores.append(f1_score(y_train[va_idx], preds, pos_label=1))

    results.append({
        **params,
        'cv_f1': np.mean(fold_scores),
        'cv_std': np.std(fold_scores)
    })

results_df = pd.DataFrame(results).sort_values('cv_f1', ascending=False).reset_index(drop=True)
print(f"Search completed in {time.time() - t0:.1f} seconds\n")
display(results_df.head(10))

best = results_df.iloc[0].to_dict()
print(f"\nBest combination found:")
print({k: best[k] for k in keys})
print(f"CV F1 (Spam) = {best['cv_f1']:.4f} ± {best['cv_std']:.4f}")

Total combinations to evaluate: 64
Search completed in 36.6 seconds



,vec_type,ngram_range,min_df,alpha,variant,cv_f1,cv_std
0,count,"(1, 1)",1,0.10,bernoulli,0.957235,0.012514
1,count,"(1, 1)",2,0.05,bernoulli,0.957235,0.012514
2,count,"(1, 1)",2,0.10,bernoulli,0.957235,0.012514
3,tfidf,"(1, 1)",1,0.10,bernoulli,0.957235,0.012514
4,tfidf,"(1, 1)",2,0.05,bernoulli,0.957235,0.012514
5,tfidf,"(1, 1)",2,0.10,bernoulli,0.957235,0.012514
6,tfidf,"(1, 1)",1,0.05,bernoulli,0.957214,0.013359
7,count,"(1, 1)",1,0.05,bernoulli,0.957214,0.013359
8,count,"(1, 1)",2,0.50,bernoulli,0.953054,0.013475
9,tfidf,"(1, 1)",2,0.50,bernoulli,0.953054,0.013475



Best combination found:
{'vec_type': 'count', 'ngram_range': (1, 1), 'min_df': 1, 'alpha': 0.1, 'variant': 'bernoulli'}
CV F1 (Spam) = 0.9572 ± 0.0125


In [33]:
VecClass = CountVectorizer if best['vec_type'] == 'count' else TfidfVectorizer
final_vectorizer = VecClass(ngram_range=best['ngram_range'], min_df=best['min_df'])

X_train_final = final_vectorizer.fit_transform(train_df['clean'])
X_test_final = final_vectorizer.transform(test_df['clean'])

final_model = CustomNaiveBayes(alpha=best['alpha'], variant=best['variant'])
final_model.fit(X_train_final, y_train)

print(f"Final Vocabulary size: {len(final_vectorizer.vocabulary_)}")
print(f"X_train shape: {X_train_final.shape} | X_test shape: {X_test_final.shape}")
print("Final model trained successfully on the full trainset.")

Final Vocabulary size: 5544
X_train shape: (4135, 5544) | X_test shape: (1034, 5544)
Final model trained successfully on the full trainset.


In [34]:
def show_final_pipeline(source_df, X_vec, y, name, n=3):
    """Traces sample messages through the final fitted feature-engineering pipeline."""
    print('=' * 75)
    print(f'FEATURE ENGINEERING – {n} {name} examples through final pipeline')
    print('=' * 75)

    names = final_vectorizer.get_feature_names_out()
    vocab = final_vectorizer.vocabulary_

    idx = np.where(y == 1)[0][:1].tolist() + np.where(y == 0)[0][:2].tolist()

    for i in idx[:n]:
        raw = source_df.Message.iloc[i]
        label = "spam" if y[i] == 1 else "ham"
        print(f'\n[True Label = {label}]')
        print(f'1. Raw Text          : {raw[:90]}')
        print(f'2. Cleaned Text      : {source_df.clean.iloc[i][:90]}')

        row = X_vec[i]
        active_indices = row.indices
        print(f'3. Vector Dimensions : {row.shape[1]} dims, {len(active_indices)} non-zero features')

        active_words = [names[j] for j in active_indices[:10]]
        print(f'4. Active Features   : {active_words}')

        oov = [w for w in source_df.clean.iloc[i].split() if w not in vocab]
        print(f'5. OOV (Dropped)     : {oov if oov else "-"}')

show_final_pipeline(train_df, X_train_final, y_train, 'TRAIN', n=3)

FEATURE ENGINEERING – 3 TRAIN examples through final pipeline

[True Label = spam]
1. Raw Text          : 18 days to Euro2004 kickoff! U will be kept informed of all the latest news and results da
2. Cleaned Text      : numtoken day euro numtoken kickoff kept inform latest news result daili unsubscrib send ge
3. Vector Dimensions : 5544 dims, 15 non-zero features
4. Active Features   : ['daili', 'day', 'euro', 'get', 'inform', 'kept', 'kickoff', 'latest', 'news', 'numtoken']
5. OOV (Dropped)     : -

[True Label = ham]
1. Raw Text          : Ta-Daaaaa! I am home babe, are you still up ?
2. Cleaned Text      : ta daaaaa home babe still
3. Vector Dimensions : 5544 dims, 5 non-zero features
4. Active Features   : ['babe', 'daaaaa', 'home', 'still', 'ta']
5. OOV (Dropped)     : -

[True Label = ham]
1. Raw Text          : Yup having my lunch buffet now.. U eat already?
2. Cleaned Text      : yup lunch buffet eat alreadi
3. Vector Dimensions : 5544 dims, 5 non-zero features
4. Active Featur

##  Prediction & Performance Evaluation on Test Set

The custom model is evaluated on the untouched **test set** using the optimal hyperparameter combination.

Below are the detailed metrics, confusion matrix, and generalization gap analysis.

In [35]:
show_final_pipeline(test_df, X_test_final, y_test, 'TEST')

FEATURE ENGINEERING – 3 TEST examples through final pipeline

[True Label = spam]
1. Raw Text          : I am hot n horny and willing I live local to you - text a reply to hear strt back from me 
2. Cleaned Text      : hot horni will live local text repli hear strt back numtoken per msg netcollex ltdhelpdesk
3. Vector Dimensions : 5544 dims, 16 non-zero features
4. Active Features   : ['back', 'end', 'hear', 'horni', 'hot', 'live', 'local', 'msg', 'netcollex', 'numtoken']
5. OOV (Dropped)     : ['strt', 'ltdhelpdesk']

[True Label = ham]
1. Raw Text          : Good morning, my Love ... I go to sleep now and wish you a great day full of feeling bette
2. Cleaned Text      : good morn love go sleep wish great day full feel better opportun last thought babe love ki
3. Vector Dimensions : 5544 dims, 16 non-zero features
4. Active Features   : ['babe', 'better', 'day', 'feel', 'full', 'go', 'good', 'great', 'kiss', 'last']
5. OOV (Dropped)     : -

[True Label = ham]
1. Raw Text          : A

In [36]:
y_pred = final_model.predict(X_test_final)
y_proba = final_model.predict_proba(X_test_final)

print('First 5 predictions on the test set:')
print('-' * 78)
for i in range(5):
    ok = 'OK ' if y_pred[i] == y_test[i] else 'ERR'
    print(f'[{ok}] true={test_df.Category.iloc[i]:<5} '
          f'pred={"spam" if y_pred[i] else "ham":<5} '
          f'P(spam)={y_proba[i, 1]:.4f} | {test_df.Message.iloc[i][:45]}')

print('\nFirst 3 spam messages in the test set:')
print('-' * 78)
for i in np.where(y_test == 1)[0][:3]:
    ok = 'OK ' if y_pred[i] == y_test[i] else 'ERR'
    print(f'[{ok}] true=spam  '
          f'pred={"spam" if y_pred[i] else "ham":<5} '
          f'P(spam)={y_proba[i, 1]:.4f} | {test_df.Message.iloc[i][:45]}')

First 5 predictions on the test set:
------------------------------------------------------------------------------
[OK ] true=ham   pred=ham   P(spam)=0.0000 | Good morning, my Love ... I go to sleep now a
[OK ] true=ham   pred=ham   P(spam)=0.0000 | And how you will do that, princess? :)
[OK ] true=ham   pred=ham   P(spam)=0.0000 | Cool, I'll text you when I'm on the way
[OK ] true=ham   pred=ham   P(spam)=0.0000 | Your right! I'll make the appointment right n
[OK ] true=ham   pred=ham   P(spam)=0.0000 | Aiya we discuss later lar... Pick ü up at 4 i

First 3 spam messages in the test set:
------------------------------------------------------------------------------
[OK ] true=spam  pred=spam  P(spam)=1.0000 | I am hot n horny and willing I live local to 
[OK ] true=spam  pred=spam  P(spam)=1.0000 | Your unique user ID is 1172. For removal send
[ERR] true=spam  pred=ham   P(spam)=0.0216 | Thanks for the Vote. Now sing along with the 


In [37]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import pandas as pd

f1_spam = f1_score(y_test, y_pred, pos_label=1)

print("=" * 60)
print(f" FINAL TEST EVALUATION | Spam Class F1-Score: {f1_spam:.4f}")
print("=" * 60)

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam'], digits=4))

cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=['Actual: Ham', 'Actual: Spam'],
    columns=['Predicted: Ham', 'Predicted: Spam']
)

print("\nClassification Breakdown (Counts):")
display(cm_df)

cv_f1 = best['cv_f1']
gap = abs(cv_f1 - f1_spam)

summary_df = pd.DataFrame({
    'Metric': ['Validation F1 (CV Mean)', 'Test Set F1', 'Generalization Gap'],
    'Score': [f"{cv_f1:.4f}", f"{f1_spam:.4f}", f"{gap:.4f}"]
})

print("\nPerformance & Generalization Summary:")
display(summary_df)

 FINAL TEST EVALUATION | Spam Class F1-Score: 0.9562

Detailed Classification Report:
              precision    recall  f1-score   support

         Ham     0.9880    1.0000    0.9939       903
        Spam     1.0000    0.9160    0.9562       131

    accuracy                         0.9894      1034
   macro avg     0.9940    0.9580    0.9751      1034
weighted avg     0.9895    0.9894    0.9892      1034


Classification Breakdown (Counts):


,Predicted: Ham,Predicted: Spam
Actual: Ham,903,0
Actual: Spam,11,120



Performance & Generalization Summary:


,Metric,Score
0,Validation F1 (CV Mean),0.9572
1,Test Set F1,0.9562
2,Generalization Gap,0.0011


## Extension: Handling Imbalanced Data

The dataset exhibits a significant class imbalance, containing substantially more ham messages than spam messages.

To investigate whether balancing the training set enhances prediction quality, Random Undersampling is applied exclusively within each training fold during 5-Fold Cross Validation.

This process reduces the number of majority class examples (ham) to match the minority class (spam), preventing data leakage during evaluation.

In [45]:
from imblearn.under_sampling import RandomUnderSampler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
undersampling_f1_scores = []

y_all = np.where(train_df['Category'] == 'spam', 1, 0)

for train_idx, val_idx in skf.split(train_df['Message'], y_all):

    X_tr_text = train_df['Message'].iloc[train_idx]
    y_tr = y_all[train_idx]

    X_val_text = train_df['Message'].iloc[val_idx]
    y_val = y_all[val_idx]

    rus = RandomUnderSampler(random_state=42)
    X_tr_resampled, y_tr_resampled = rus.fit_resample(
        X_tr_text.values.reshape(-1, 1),
        y_tr
    )
    X_tr_resampled = X_tr_resampled.ravel()

    vectorizer = CountVectorizer()
    X_tr_vec = vectorizer.fit_transform(X_tr_resampled)
    X_val_vec = vectorizer.transform(X_val_text)

    clf = MultinomialNB()
    clf.fit(X_tr_vec, y_tr_resampled)

    preds = clf.predict(X_val_vec)
    f1 = f1_score(y_val, preds, pos_label=1)
    undersampling_f1_scores.append(f1)

undersampling_mean_f1 = np.mean(undersampling_f1_scores)

baseline_f1 = best_cv_f1 if 'best_cv_f1' in globals() else 0.947

comparison_df = pd.DataFrame({
    'Method': ['Original Data (Imbalanced)', 'Random Undersampling'],
    'Mean Validation F1': [baseline_f1, undersampling_mean_f1]
})

print("--- Part 6B: Handling Imbalanced Data ---")
display(comparison_df)

--- Part 6B: Handling Imbalanced Data ---


,Method,Mean Validation F1
0,Original Data (Imbalanced),0.947000
1,Random Undersampling,0.849281


## Extension: Model Explainability & Feature Importance

To understand how the Naive Bayes model makes its decisions, we analyze the feature log-odds ratios.

By comparing the log-probabilities of words given the **Spam** class versus the **Ham** class, we identify the top terms that serve as the strongest predictors for each category.

$$\text{Log-Odds Ratio}(w) = \ln P(w \mid \text{Spam}) - \ln P(w \mid \text{Ham})$$

Words with the highest positive scores are the strongest indicators of Spam.

In [46]:
import numpy as np
import pandas as pd

final_vectorizer = CountVectorizer()
X_full_train = final_vectorizer.fit_transform(train_df['Message'])
y_full_train = np.where(train_df['Category'] == 'spam', 1, 0)

final_model = MultinomialNB()
final_model.fit(X_full_train, y_full_train)

log_prob_ham = final_model.feature_log_prob_[0]
log_prob_spam = final_model.feature_log_prob_[1]

log_odds_ratios = log_prob_spam - log_prob_ham
feature_names = final_vectorizer.get_feature_names_out()

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Log_Odds_Ratio': log_odds_ratios
})

top_spam = importance_df.sort_values(by='Log_Odds_Ratio', ascending=False).head(10).reset_index(drop=True)
top_ham = importance_df.sort_values(by='Log_Odds_Ratio', ascending=True).head(10).reset_index(drop=True)

print("TOP 10 SPAM INDICATORS")
display(top_spam)

print("\nTOP 10 HAM INDICATORS")
display(top_ham)

TOP 10 SPAM INDICATORS


,Feature,Log_Odds_Ratio
0,claim,5.304820
1,prize,5.204016
2,150p,4.798550
3,co,4.625279
4,tone,4.625279
5,www,4.583894
6,18,4.569709
7,500,4.480097
8,guaranteed,4.480097
9,1000,4.381657



TOP 10 HAM INDICATORS


,Feature,Log_Odds_Ratio
0,gt,-4.370134
1,lt,-4.360918
2,he,-4.173025
3,lor,-3.905620
4,da,-3.789660
5,she,-3.649078
6,later,-3.329445
7,but,-3.211012
8,really,-3.205147
9,doing,-3.205147


### Analysis & Interpretation

The Log-Odds Ratio analysis successfully reveals the core word associations learned by the Naive Bayes model:

* **Top Spam Indicators:** Words like `claim`, `prize`, `guaranteed`, and numerical values (e.g., `1000`, `150p`) have high positive log-odds ratios. This indicates that their presence strongly pushes the model's decision toward classifying a message as **Spam**, directly mirroring typical promotional and scam patterns.
* **Top Ham Indicators:** Words such as `later`, `really`, and `doing` yield negative log-odds ratios, indicating they are characteristic of personal, informal everyday conversations (**Ham**).

This explainability approach provides transparency into the model's decision-making process without relying on complex black-box tools.